In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.providers.postgres.hooks.postgres import PostgresHook
from datetime import datetime
import sqlite3
import csv
import pandas as pd
import os

# --- НАСТРОЙКИ ПУТЕЙ (ДЛЯ WSL) ---
# ВАЖНО: Файлы CSV должны лежать ВНУТРИ WSL, а не на Windows Desktop.
# Создай папку: mkdir -p ~/my-project/dags/data
# И положи туда DATA Sales.csv и DATA Margin.csv
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(BASE_DIR, "data")

SALES_CSV = os.path.join(DATA_DIR, "DATA Sales.csv")
MARGIN_CSV = os.path.join(DATA_DIR, "DATA Margin.csv")
DB_PATH = os.path.join(BASE_DIR, "alfor.db")

def extract_data_sales(**kwargs):
    print(f"Читаем продажи из: {SALES_CSV}")
    if not os.path.exists(SALES_CSV):
        raise FileNotFoundError(f"Файл не найден: {SALES_CSV}. Проверь папку data!")

    db = sqlite3.connect(DB_PATH)
    c = db.cursor()
    
    # Создаем таблицу, если нет
    c.execute("""
        CREATE TABLE IF NOT EXISTS sales(
            date_week TEXT,
            item INTEGER,
            item_name TEXT,
            store INTEGER,
            sales_qty REAL
        )
    """)
    c.execute("DELETE FROM sales")

    with open(SALES_CSV, newline="", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter=";")
        next(reader, None)  # Пропуск заголовка
        for row in reader:
            if len(row) >= 5:
                c.execute(
                    "INSERT INTO sales VALUES (?, ?, ?, ?, ?)",
                    row[:5]
                )
    db.commit()
    db.close()
    return "sales_ok"


def extract_data_margin(**kwargs):
    print(f"Читаем маржу из: {MARGIN_CSV}")
    if not os.path.exists(MARGIN_CSV):
        raise FileNotFoundError(f"Файл не найден: {MARGIN_CSV}. Проверь папку data!")

    db = sqlite3.connect(DB_PATH)
    c = db.cursor()

    c.execute("""
        CREATE TABLE IF NOT EXISTS margin(
            date_week TEXT,
            item INTEGER,
            item_name TEXT,
            store INTEGER,
            profit_rub REAL
        )
    """)
    c.execute("DELETE FROM margin")

    with open(MARGIN_CSV, newline="", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter=";")
        next(reader, None)
        for row in reader:
            if len(row) >= 5:
                c.execute(
                    "INSERT INTO margin VALUES (?, ?, ?, ?, ?)",
                    row[:5]
                )
    db.commit()
    db.close()
    return "margin_ok"


def transform_and_load(**kwargs):
    print("Начинаем трансформацию и загрузку в Postgres...")
    
    # 1. Читаем из SQLite
    conn = sqlite3.connect(DB_PATH)
    query = """
        SELECT
            t1.date_week,
            t1.item,
            t1.item_name,
            t1.store,
            t1.sales_qty,
            t2.profit_rub
        FROM sales AS t1
        LEFT JOIN margin AS t2
            ON t1.date_week = t2.date_week
           AND t1.item    = t2.item
           AND t1.store   = t2.store
    """
    df = pd.read_sql_query(query, conn)
    conn.close()

    if df.empty:
        print("Предупреждение: DataFrame пуст! Ничего не будем грузить.")
        return "empty_df"

    print(f"Сформировано строк для загрузки: {len(df)}")

    # 2. Подключаемся к Postgres
    pg_hook = PostgresHook(postgres_conn_id="my_postgres_conn")
    
    try:
        # ВАЖНО: Используем engine из hook, чтобы избежать проблем с psycopg3
        engine = pg_hook.get_sqlalchemy_engine()
        
        # to_sql теперь работает корректно через SQLAlchemy
        # if_exists='replace' -> удалит старую таблицу и создаст новую
        df.to_sql(
            name='company_result',
            con=engine,
            schema=None,
            if_exists='replace', 
            index=False,
            chunksize=1000,
            method='multi'
        )
        
        print("Данные успешно загружены в таблицу company_result в Postgres!")
        return "load_ok"

    except Exception as e:
        print(f"Критическая ошибка при загрузке: {e}")
        # Пробрасываем ошибку дальше, чтобы задача упала в UI Airflow
        raise e

def run_data_quality_checks(**context):
    """
    Проверяет качество данных в загруженных таблицах.
    Падает с ошибкой, если данные плохие.
    """
    conn_id = 'my_postgres_conn' # Твое соединение из Admin -> Connections
    hook = PostgresHook(postgres_conn_id=conn_id)

       # --- ПРОВЕРКА 1: Количество строк (Completeness) ---
    # Допустим, мы ожидаем, что в таблице sales больше 0 строк
    count_sales = hook.get_first("SELECT COUNT(*) FROM sales")[0]
    if count_sales == 0:
        raise ValueError(f"CRITICAL: Таблица sales пуста! Загружено 0 строк.")
    print(f"✅ Проверка 1 пройдена: В таблице sales найдено {count_sales} строк.")

with DAG(
    dag_id="company_result_fixed",  # Новое имя, чтобы не путаться со старым сломанным
    start_date=datetime(2024, 1, 1),  # Дата в ПРОШЛОМ! Иначе DAG не запустится.
    schedule='@daily',  # Запускаем вручную для теста
    catchup=False,
    tags=['etl', 'sales'],
) as dag:

    extract_sales = PythonOperator(
        task_id="extract_data_sales",
        python_callable=extract_data_sales,
    )

    extract_margin = PythonOperator(
        task_id="extract_data_margin",
        python_callable=extract_data_margin,
    )

    # ИСПРАВЛЕНО: python_callable вместо ошибочного python_camy_postgres_conn
    transform_load = PythonOperator(
        task_id="transform_and_load_to_postgres",
        python_callable=transform_and_load,
    )

     # НОВАЯ ЗАДАЧА: Проверка качества
    check_quality = PythonOperator(
        task_id='check_quality',
        python_callable=run_data_quality_checks,
        provide_context=True, # Нужно для старых версий, в новых можно убрать
        # depends_on_past=False # По умолчанию
    )
    # УБРАЛИ PostgresOperator с CREATE TABLE и COPY.
    # Теперь вся логика создания таблицы и вставки данных находится внутри transform_load (через pandas).
    # Это единственный рабочий способ в Airflow без внешних скриптов.

    # Цепочка задач
    extract_sales >> extract_margin >> transform_load >> check_quality